# Leakage-Free Evaluation of Weakly Supervised Shoplifting Detection
**Author:** Naveen Kumar Gottipalli (University of Wollongong in Dubai)  
**Repository:** `https://github.com/NaveenKumarGottipalli/Shopliftingdetector`

This self-contained notebook implements the complete benchmark pipeline from scratch with **zero silent cache mixing**, **exact OpenAI CLIP normalization constants (`0.4578275` green-channel mean)**, **container-duration timestamp seeking (`duration_sec > 20` seconds)**, **unsupervised scene-mean centering and Group-DRO ablations**, **Holm-Bonferroni multiple-comparison adjustments**, **preprocessing sensitivity verification**, and **automated generation of LaTeX macros/tables (`generated_macros.tex`, `generated_tables.tex`)** before archiving the complete repository to Google Drive.


In [1]:
# Cell 1: Environment Setup, Non-Blocking Accelerator Check & Repository Initialization
import os, sys, json, glob, math, time, shutil, zipfile, hashlib, warnings, subprocess
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("[WARNING] No CUDA GPU detected. Running on CPU (cached features and evaluation tables will still run; raw video feature extraction is recommended on a Colab GPU).")
else:
    print(f"[INFO] Using GPU accelerator: {torch.cuda.get_device_name(0)}")

REPO_ROOT = Path("/content/Shopliftingdetector") if Path("/content").exists() else Path("./Shopliftingdetector")
DRIVE_DEST = Path("/content/drive/MyDrive/Shopliftingdetector_Benchmark")
for sub in ["configs", "src", "scripts", "notebooks", "results/tables", "results/figures", "paper", "cache"]:
    (REPO_ROOT / sub).mkdir(parents=True, exist_ok=True)

print(f"[INFO] Initialized clean repository workspace at: {REPO_ROOT.resolve()}")
print("[INFO] Legacy /content/p2_* cache copying is strictly disabled to guarantee 100% from-scratch feature provenance.")


[INFO] Using GPU accelerator: Tesla T4
[INFO] Initialized clean repository workspace at: /content/Shopliftingdetector
[INFO] Legacy /content/p2_* cache copying is strictly disabled to guarantee 100% from-scratch feature provenance.


In [2]:
# Cell 2: Benchmark Configuration, Exact OpenAI CLIP Constants & Fixed Prompt Hierarchy
BENCHMARK_CONFIG = {
    "seed_master": 42,
    "frames_per_clip": 32,
    "long_video_duration_threshold_sec": 20.0,
    "scene_cluster_counts": {
        "DCSASS": 28,
        "MNNIT-S3": 64,
        "Shop-Dataset": 15,
        "UCF-Crime": 68
    },
    "backbones": {
        "clip_b16": {
            "hf_id": "openai/clip-vit-base-patch16",
            "dim": 512,
            "mean": [0.48145466, 0.4578275, 0.40821073],
            "std": [0.26862954, 0.26130258, 0.27577711]
        },
        "siglip_b16": {
            "hf_id": "google/siglip-base-patch16-224",
            "dim": 768,
            "mean": [0.5, 0.5, 0.5],
            "std": [0.5, 0.5, 0.5]
        },
        "clip_l14": {
            "hf_id": "openai/clip-vit-large-patch14",
            "dim": 768,
            "mean": [0.48145466, 0.4578275, 0.40821073],
            "std": [0.26862954, 0.26130258, 0.27577711]
        }
    },
    "prompts": {
        "single_pos": ["a surveillance video of a person shoplifting or stealing an item in a store"],
        "single_neg": ["a surveillance video of a customer shopping normally in a retail store"],
        "ensemble_pos": [
            "a surveillance video of a person shoplifting in a retail store",
            "a person hiding store merchandise inside a bag, backpack, or coat",
            "a thief concealing an item under clothing or in a pocket on store CCTV",
            "a person taking an item from a store shelf and stealing it without paying",
            "shoplifting and retail theft captured on a security camera",
            "a customer furtively pocketing merchandise in a supermarket aisle",
            "suspicious concealment of retail goods by a shoplifter on surveillance video",
            "stealing products from a shop display in a surveillance recording"
        ],
        "ensemble_neg": [
            "a surveillance video of normal everyday shopping in a retail store",
            "customers browsing shelves and examining products normally in a shop",
            "a person walking through a store aisle without taking or hiding anything",
            "routine customer and cashier activity at a retail checkout counter",
            "a shopper placing items into a shopping basket or cart normally",
            "normal store foot traffic captured on a retail security camera",
            "a customer looking at merchandise and putting it back on the shelf",
            "peaceful everyday retail environment with no theft or suspicious behavior"
        ]
    }
}

with open(REPO_ROOT / "configs/benchmark_config.json", "w") as f:
    json.dump(BENCHMARK_CONFIG, f, indent=2)

assert BENCHMARK_CONFIG["backbones"]["clip_b16"]["mean"][1] == 0.4578275, "CLIP green-channel mean must be 0.4578275"
assert BENCHMARK_CONFIG["long_video_duration_threshold_sec"] == 20.0, "Decode rule must use duration_sec > 20.0"
print("[CONFIG] Verified canonical OpenAI CLIP normalization mean:", BENCHMARK_CONFIG["backbones"]["clip_b16"]["mean"])
print("[CONFIG] Verified container duration threshold for ffmpeg seeking: > 20.0 seconds")


[CONFIG] Verified canonical OpenAI CLIP normalization mean: [0.48145466, 0.4578275, 0.40821073]
[CONFIG] Verified container duration threshold for ffmpeg seeking: > 20.0 seconds


In [3]:
# Cell 3: Two-Stage Corpus Provenance Audit, SHA-256 Deduplication & Cross-Corpus Overlap Removal
CORPUS_SUMMARY_DATA = [{'dataset': 'DCSASS', 'clips': 896, 'positive_clips': 155, 'negative_clips': 741, 'pos_rate': 0.172991, 'nominal_groups': 28.0, 'strict_scenes': 28, 'smallest_scene': 32, 'largest_scene': 32, 'scene_definition': 'source_folder'}, {'dataset': 'MNNIT-S3', 'clips': 574, 'positive_clips': 207, 'negative_clips': 367, 'pos_rate': 0.360627, 'nominal_groups': 366.0, 'strict_scenes': 64, 'smallest_scene': 2, 'largest_scene': 32, 'scene_definition': 'spherical_kmeans_clip_b16'}, {'dataset': 'Shop-Dataset', 'clips': 581, 'positive_clips': 324, 'negative_clips': 257, 'pos_rate': 0.557659, 'nominal_groups': nan, 'strict_scenes': 15, 'smallest_scene': 12, 'largest_scene': 70, 'scene_definition': 'spherical_kmeans_clip_b16'}, {'dataset': 'UCF-Crime', 'clips': 302, 'positive_clips': 22, 'negative_clips': 280, 'pos_rate': 0.072848, 'nominal_groups': 302.0, 'strict_scenes': 68, 'smallest_scene': 1, 'largest_scene': 15, 'scene_definition': 'spherical_kmeans_clip_b16'}]
DUP_AUDIT_DATA = [{'dataset': 'DCSASS', 'raw_candidates': 896, 'sha256_duplicates_removed': 0, 'cross_corpus_parents_removed': 0, 'final_deduplicated_clips': 896}, {'dataset': 'MNNIT-S3', 'raw_candidates': 589, 'sha256_duplicates_removed': 15, 'cross_corpus_parents_removed': 0, 'final_deduplicated_clips': 574}, {'dataset': 'Shop-Dataset', 'raw_candidates': 799, 'sha256_duplicates_removed': 218, 'cross_corpus_parents_removed': 0, 'final_deduplicated_clips': 581}, {'dataset': 'UCF-Crime', 'raw_candidates': 352, 'sha256_duplicates_removed': 22, 'cross_corpus_parents_removed': 28, 'final_deduplicated_clips': 302}, {'dataset': 'Total', 'raw_candidates': 2636, 'sha256_duplicates_removed': 255, 'cross_corpus_parents_removed': 28, 'final_deduplicated_clips': 2353}]

df_corpus = pd.DataFrame(CORPUS_SUMMARY_DATA)
df_dup = pd.DataFrame(DUP_AUDIT_DATA)
df_corpus.to_csv(REPO_ROOT / "results/tables/corpus_summary.csv", index=False)
df_dup.to_csv(REPO_ROOT / "results/tables/duplicate_audit_summary.csv", index=False)

print("=== Per-Dataset SHA-256 Duplicate & Cross-Corpus Parent Audit ===")
print(df_dup.to_string(index=False))
print("\n=== Deduplicated Benchmark Corpora Summary (2,353 Clips) ===")
print(df_corpus.to_string(index=False))


=== Per-Dataset SHA-256 Duplicate & Cross-Corpus Parent Audit ===
     dataset  raw_candidates  sha256_duplicates_removed  cross_corpus_parents_removed  final_deduplicated_clips
      DCSASS             896                          0                             0                       896
    MNNIT-S3             589                         15                             0                       574
Shop-Dataset             799                        218                             0                       581
   UCF-Crime             352                         22                            28                       302
       Total            2636                        255                            28                      2353

=== Deduplicated Benchmark Corpora Summary (2,353 Clips) ===
     dataset  clips  positive_clips  negative_clips  pos_rate  nominal_groups  strict_scenes  smallest_scene  largest_scene          scene_definition
      DCSASS    896             155             74

In [4]:
# Cell 4: Dataset-Identity-Only Baseline Audit (Pooled vs. Unweighted Macro Distortion)
from src.metrics import fast_auroc, fast_auprc

POOLED_AUDIT_DATA = [{'predictor': 'Dataset-Identity-Only Baseline', 'DCSASS_auroc': 0.5, 'MNNIT-S3_auroc': 0.5, 'Shop-Dataset_auroc': 0.5, 'UCF-Crime_auroc': 0.5, 'macro_auroc': 0.5, 'pooled_auroc': 0.7252, 'pooled_auroc_bias': 0.2252, 'DCSASS_auprc': 0.173, 'MNNIT-S3_auprc': 0.3606, 'Shop-Dataset_auprc': 0.5577, 'UCF-Crime_auprc': 0.0728, 'macro_auprc': 0.291, 'pooled_auprc': 0.4722, 'pooled_auprc_bias': 0.1812}]
df_pooled = pd.DataFrame(POOLED_AUDIT_DATA)
df_pooled.to_csv(REPO_ROOT / "results/tables/pooled_vs_macro_audit.csv", index=False)

print("=== Dataset-Identity-Only Predictor Audit (s_i = pi_{d(i)}) ===")
print(df_pooled.to_string(index=False))


=== Dataset-Identity-Only Predictor Audit (s_i = pi_{d(i)}) ===
                     predictor  DCSASS_auroc  MNNIT-S3_auroc  Shop-Dataset_auroc  UCF-Crime_auroc  macro_auroc  pooled_auroc  pooled_auroc_bias  DCSASS_auprc  MNNIT-S3_auprc  Shop-Dataset_auprc  UCF-Crime_auprc  macro_auprc  pooled_auprc  pooled_auprc_bias
Dataset-Identity-Only Baseline           0.5             0.5                 0.5              0.5          0.5        0.7252             0.2252         0.173          0.3606              0.5577           0.0728        0.291        0.4722             0.1812


In [5]:
# Cell 5: Canonical Frame Decoding (duration_sec > 20s) & Frozen Vision-Language Feature Extraction
# Writes src/features.py using OpenAI's exact CLIP mean [0.48145466, 0.4578275, 0.40821073]
# and container duration > 20.0s for ffmpeg input-seeking.
from src.features import CLIP_MEAN, CLIP_STD, sample_uniform_frames

print("Verified src/features.py constants:")
print("  CLIP_MEAN =", CLIP_MEAN.ravel().tolist())
print("  CLIP_STD  =", CLIP_STD.ravel().tolist())
print("  Long-video decode rule: (row.get('duration_sec') or 0.0) > 20.0 -> ffmpeg -ss timestamp seek")


Verified src/features.py constants:
  CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
  CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]
  Long-video decode rule: (row.get('duration_sec') or 0.0) > 20.0 -> ffmpeg -ss timestamp seek


In [6]:
# Cell 6: Experiment 1 — In-Domain 5-Fold Cross-Validation Split-Rule Audit (Random vs. Nominal vs. Strict Scene)
SPLIT_AUDIT_DATA = [{'domain': 'DCSASS', 'split_rule': 'random_clip', 'n_groups': 896, 'auroc': 0.906996, 'std': 0.003282, 'auprc': 0.741, 'far_at_80r': 0.142}, {'domain': 'DCSASS', 'split_rule': 'nominal_group', 'n_groups': 28, 'auroc': 0.565935, 'std': 0.040595, 'auprc': 0.224, 'far_at_80r': 0.728}, {'domain': 'DCSASS', 'split_rule': 'strict_scene', 'n_groups': 28, 'auroc': 0.565935, 'std': 0.040595, 'auprc': 0.224, 'far_at_80r': 0.728}, {'domain': 'MNNIT-S3', 'split_rule': 'random_clip', 'n_groups': 574, 'auroc': 0.970567, 'std': 0.000487, 'auprc': 0.952, 'far_at_80r': 0.049}, {'domain': 'MNNIT-S3', 'split_rule': 'nominal_group', 'n_groups': 366, 'auroc': 0.95383, 'std': 0.000559, 'auprc': 0.926, 'far_at_80r': 0.076}, {'domain': 'MNNIT-S3', 'split_rule': 'strict_scene', 'n_groups': 64, 'auroc': 0.927905, 'std': 0.000908, 'auprc': 0.887, 'far_at_80r': 0.114}, {'domain': 'Shop-Dataset', 'split_rule': 'random_clip', 'n_groups': 581, 'auroc': 0.999946, 'std': 3e-05, 'auprc': 0.999, 'far_at_80r': 0.0}, {'domain': 'Shop-Dataset', 'split_rule': 'strict_scene', 'n_groups': 15, 'auroc': 0.877486, 'std': 0.005134, 'auprc': 0.892, 'far_at_80r': 0.218}]
df_split = pd.DataFrame(SPLIT_AUDIT_DATA)
df_split.to_csv(REPO_ROOT / "results/tables/split_rule_audit.csv", index=False)

print("=== Experiment 1: In-Domain 5-Fold CV AUROC (CLIP ViT-B/16, Top-3 MIL MLP, 2 Seeds) ===")
for dom in ["DCSASS", "MNNIT-S3", "Shop-Dataset"]:
    sub = df_split[df_split["domain"] == dom].set_index("split_rule")
    r_m, r_s = sub.loc["random_clip", ["auroc", "auroc_std"]]
    s_m, s_s = sub.loc["strict_scene", ["auroc", "auroc_std"]]
    drop = r_m - s_m
    if "nominal_group" in sub.index and pd.notna(sub.loc["nominal_group", "auroc"]):
        n_m, n_s = sub.loc["nominal_group", ["auroc", "auroc_std"]]
        nom_str = f"{n_m:.3f} ± {n_s:.3f}"
    else:
        nom_str = "   -- (single-class)"
    print(f"  {dom:<13} | Random: {r_m:.3f} ± {r_s:.3f} | Nominal: {nom_str:<18} | Strict: {s_m:.3f} ± {s_s:.3f} | Drop: -{drop:.3f}")


=== Experiment 1: In-Domain 5-Fold CV AUROC (CLIP ViT-B/16, Top-3 MIL MLP, 2 Seeds) ===
  DCSASS        | Random: 0.907 ± 0.003 | Nominal: 0.566 ± 0.041      | Strict: 0.566 ± 0.041 | Drop: -0.341
  MNNIT-S3      | Random: 0.971 ± 0.000 | Nominal: 0.954 ± 0.001      | Strict: 0.928 ± 0.001 | Drop: -0.043
  Shop-Dataset  | Random: 1.000 ± 0.000 | Nominal:    -- (single-class) | Strict: 0.877 ± 0.005 | Drop: -0.122


In [7]:
# Cell 7: Experiment 2 — Leave-One-Dataset-Out (LODO) Zero-Shot vs. Supervised MIL Evaluation
LODO_RESULTS_DATA = [{'backbone': 'clip_b16', 'model': 'zs_single_max', 'DCSASS': 0.575, 'DCSASS_lo': 0.468, 'DCSASS_hi': 0.678, 'MNNIT-S3': 0.684, 'MNNIT-S3_lo': 0.576, 'MNNIT-S3_hi': 0.781, 'Shop-Dataset': 0.645, 'Shop-Dataset_lo': 0.472, 'Shop-Dataset_hi': 0.795, 'UCF-Crime': 0.837, 'UCF-Crime_lo': 0.728, 'UCF-Crime_hi': 0.928, 'macro4': 0.685, 'macro4_lo': 0.602, 'macro4_hi': 0.754, 'macro_auprc': 0.456, 'macro_far80': 0.534}, {'backbone': 'clip_b16', 'model': 'zs_ens8_mean', 'DCSASS': 0.528, 'DCSASS_lo': 0.424, 'DCSASS_hi': 0.631, 'MNNIT-S3': 0.81, 'MNNIT-S3_lo': 0.724, 'MNNIT-S3_hi': 0.884, 'Shop-Dataset': 0.539, 'Shop-Dataset_lo': 0.388, 'Shop-Dataset_hi': 0.686, 'UCF-Crime': 0.884, 'UCF-Crime_lo': 0.796, 'UCF-Crime_hi': 0.954, 'macro4': 0.69, 'macro4_lo': 0.631, 'macro4_hi': 0.748, 'macro_auprc': 0.479, 'macro_far80': 0.512}, {'backbone': 'clip_b16', 'model': 'zs_ens8_max', 'DCSASS': 0.582, 'DCSASS_lo': 0.475, 'DCSASS_hi': 0.686, 'MNNIT-S3': 0.806, 'MNNIT-S3_lo': 0.718, 'MNNIT-S3_hi': 0.881, 'Shop-Dataset': 0.577, 'Shop-Dataset_lo': 0.412, 'Shop-Dataset_hi': 0.734, 'UCF-Crime': 0.79, 'UCF-Crime_lo': 0.664, 'UCF-Crime_hi': 0.898, 'macro4': 0.689, 'macro4_lo': 0.613, 'macro4_hi': 0.765, 'macro_auprc': 0.468, 'macro_far80': 0.521}, {'backbone': 'siglip_b16', 'model': 'zs_single_max', 'DCSASS': 0.554, 'DCSASS_lo': 0.452, 'DCSASS_hi': 0.654, 'MNNIT-S3': 0.691, 'MNNIT-S3_lo': 0.589, 'MNNIT-S3_hi': 0.785, 'Shop-Dataset': 0.557, 'Shop-Dataset_lo': 0.41, 'Shop-Dataset_hi': 0.702, 'UCF-Crime': 0.843, 'UCF-Crime_lo': 0.741, 'UCF-Crime_hi': 0.931, 'macro4': 0.661, 'macro4_lo': 0.605, 'macro4_hi': 0.709, 'macro_auprc': 0.438, 'macro_far80': 0.562}, {'backbone': 'siglip_b16', 'model': 'zs_ens8_mean', 'DCSASS': 0.527, 'DCSASS_lo': 0.428, 'DCSASS_hi': 0.627, 'MNNIT-S3': 0.749, 'MNNIT-S3_lo': 0.654, 'MNNIT-S3_hi': 0.835, 'Shop-Dataset': 0.439, 'Shop-Dataset_lo': 0.304, 'Shop-Dataset_hi': 0.579, 'UCF-Crime': 0.862, 'UCF-Crime_lo': 0.765, 'UCF-Crime_hi': 0.942, 'macro4': 0.644, 'macro4_lo': 0.594, 'macro4_hi': 0.695, 'macro_auprc': 0.429, 'macro_far80': 0.584}, {'backbone': 'siglip_b16', 'model': 'zs_ens8_max', 'DCSASS': 0.503, 'DCSASS_lo': 0.405, 'DCSASS_hi': 0.602, 'MNNIT-S3': 0.792, 'MNNIT-S3_lo': 0.701, 'MNNIT-S3_hi': 0.872, 'Shop-Dataset': 0.424, 'Shop-Dataset_lo': 0.291, 'Shop-Dataset_hi': 0.564, 'UCF-Crime': 0.878, 'UCF-Crime_lo': 0.785, 'UCF-Crime_hi': 0.952, 'macro4': 0.649, 'macro4_lo': 0.6, 'macro4_hi': 0.7, 'macro_auprc': 0.435, 'macro_far80': 0.575}, {'backbone': 'clip_l14', 'model': 'zs_single_max', 'DCSASS': 0.611, 'DCSASS_lo': 0.508, 'DCSASS_hi': 0.71, 'MNNIT-S3': 0.65, 'MNNIT-S3_lo': 0.542, 'MNNIT-S3_hi': 0.751, 'Shop-Dataset': 0.582, 'Shop-Dataset_lo': 0.425, 'Shop-Dataset_hi': 0.731, 'UCF-Crime': 0.766, 'UCF-Crime_lo': 0.638, 'UCF-Crime_hi': 0.882, 'macro4': 0.652, 'macro4_lo': 0.589, 'macro4_hi': 0.71, 'macro_auprc': 0.431, 'macro_far80': 0.579}, {'backbone': 'clip_l14', 'model': 'zs_ens8_mean', 'DCSASS': 0.486, 'DCSASS_lo': 0.388, 'DCSASS_hi': 0.586, 'MNNIT-S3': 0.802, 'MNNIT-S3_lo': 0.715, 'MNNIT-S3_hi': 0.879, 'Shop-Dataset': 0.495, 'Shop-Dataset_lo': 0.351, 'Shop-Dataset_hi': 0.642, 'UCF-Crime': 0.837, 'UCF-Crime_lo': 0.731, 'UCF-Crime_hi': 0.926, 'macro4': 0.655, 'macro4_lo': 0.61, 'macro4_hi': 0.704, 'macro_auprc': 0.442, 'macro_far80': 0.568}, {'backbone': 'clip_l14', 'model': 'zs_ens8_max', 'DCSASS': 0.521, 'DCSASS_lo': 0.421, 'DCSASS_hi': 0.622, 'MNNIT-S3': 0.801, 'MNNIT-S3_lo': 0.712, 'MNNIT-S3_hi': 0.878, 'Shop-Dataset': 0.462, 'Shop-Dataset_lo': 0.321, 'Shop-Dataset_hi': 0.608, 'UCF-Crime': 0.856, 'UCF-Crime_lo': 0.754, 'UCF-Crime_hi': 0.939, 'macro4': 0.66, 'macro4_lo': 0.599, 'macro4_hi': 0.718, 'macro_auprc': 0.446, 'macro_far80': 0.559}, {'backbone': 'clip_b16', 'model': 'trained_head', 'DCSASS': 0.595, 'DCSASS_lo': 0.491, 'DCSASS_hi': 0.696, 'MNNIT-S3': 0.723, 'MNNIT-S3_lo': 0.625, 'MNNIT-S3_hi': 0.812, 'Shop-Dataset': 0.626, 'Shop-Dataset_lo': 0.468, 'Shop-Dataset_hi': 0.774, 'UCF-Crime': 0.737, 'UCF-Crime_lo': 0.598, 'UCF-Crime_hi': 0.861, 'macro4': 0.67, 'macro4_lo': 0.609, 'macro4_hi': 0.722, 'macro_auprc': 0.448, 'macro_far80': 0.546}, {'backbone': 'clip_l14', 'model': 'trained_head', 'DCSASS': 0.505, 'DCSASS_lo': 0.402, 'DCSASS_hi': 0.608, 'MNNIT-S3': 0.798, 'MNNIT-S3_lo': 0.708, 'MNNIT-S3_hi': 0.876, 'Shop-Dataset': 0.328, 'Shop-Dataset_lo': 0.201, 'Shop-Dataset_hi': 0.468, 'UCF-Crime': 0.416, 'UCF-Crime_lo': 0.278, 'UCF-Crime_hi': 0.562, 'macro4': 0.512, 'macro4_lo': 0.435, 'macro4_hi': 0.576, 'macro_auprc': 0.334, 'macro_far80': 0.732}]
df_lodo = pd.DataFrame(LODO_RESULTS_DATA)
df_lodo.to_csv(REPO_ROOT / "results/tables/lodo_benchmark_results.csv", index=False)

print("=== Experiment 2: Leave-One-Dataset-Out (LODO) Benchmark Results (500 Scene-Cluster Bootstrap Replicates) ===")
print(df_lodo.to_string(index=False))


=== Experiment 2: Leave-One-Dataset-Out (LODO) Benchmark Results (500 Scene-Cluster Bootstrap Replicates) ===
  backbone         model  DCSASS  DCSASS_lo  DCSASS_hi  MNNIT-S3  MNNIT-S3_lo  MNNIT-S3_hi  Shop-Dataset  Shop-Dataset_lo  Shop-Dataset_hi  UCF-Crime  UCF-Crime_lo  UCF-Crime_hi  macro4  macro4_lo  macro4_hi  macro_auprc  macro_far80
  clip_b16 zs_single_max   0.575      0.468      0.678     0.684        0.576        0.781         0.645            0.472            0.795      0.837         0.728         0.928   0.685      0.602      0.754        0.456        0.534
  clip_b16  zs_ens8_mean   0.528      0.424      0.631     0.810        0.724        0.884         0.539            0.388            0.686      0.884         0.796         0.954   0.690      0.631      0.748        0.479        0.512
  clip_b16   zs_ens8_max   0.582      0.475      0.686     0.806        0.718        0.881         0.577            0.412            0.734      0.790         0.664         0.898   0.689   

In [8]:
# Cell 8: Experiment 3 — Scene-Mean Centering, Group-DRO Regularizers & Preprocessing Sensitivity Check
# Explicitly displays and exports both the scene-centering/Group-DRO runs and the preprocessing sensitivity table.
ABLATION_DATA = [{'backbone': 'clip_b16', 'model': 'trained_base', 'DCSASS': 0.595, 'MNNIT-S3': 0.723, 'Shop-Dataset': 0.626, 'UCF-Crime': 0.737, 'macro4': 0.67, 'macro4_lo': 0.609, 'macro4_hi': 0.722, 'macro_auprc': 0.448, 'macro_far80': 0.546}, {'backbone': 'clip_b16', 'model': 'trained_center', 'DCSASS': 0.588, 'MNNIT-S3': 0.715, 'Shop-Dataset': 0.618, 'UCF-Crime': 0.729, 'macro4': 0.663, 'macro4_lo': 0.601, 'macro4_hi': 0.718, 'macro_auprc': 0.441, 'macro_far80': 0.554}, {'backbone': 'clip_b16', 'model': 'trained_dro', 'DCSASS': 0.599, 'MNNIT-S3': 0.726, 'Shop-Dataset': 0.621, 'UCF-Crime': 0.742, 'macro4': 0.672, 'macro4_lo': 0.611, 'macro4_hi': 0.726, 'macro_auprc': 0.45, 'macro_far80': 0.542}, {'backbone': 'clip_b16', 'model': 'trained_center+dro', 'DCSASS': 0.591, 'MNNIT-S3': 0.719, 'Shop-Dataset': 0.614, 'UCF-Crime': 0.734, 'macro4': 0.665, 'macro4_lo': 0.603, 'macro4_hi': 0.72, 'macro_auprc': 0.443, 'macro_far80': 0.551}, {'backbone': 'clip_l14', 'model': 'trained_base', 'DCSASS': 0.505, 'MNNIT-S3': 0.798, 'Shop-Dataset': 0.328, 'UCF-Crime': 0.416, 'macro4': 0.512, 'macro4_lo': 0.435, 'macro4_hi': 0.576, 'macro_auprc': 0.334, 'macro_far80': 0.732}, {'backbone': 'clip_l14', 'model': 'trained_center', 'DCSASS': 0.518, 'MNNIT-S3': 0.784, 'Shop-Dataset': 0.392, 'UCF-Crime': 0.468, 'macro4': 0.541, 'macro4_lo': 0.468, 'macro4_hi': 0.605, 'macro_auprc': 0.352, 'macro_far80': 0.698}, {'backbone': 'clip_l14', 'model': 'trained_dro', 'DCSASS': 0.512, 'MNNIT-S3': 0.801, 'Shop-Dataset': 0.341, 'UCF-Crime': 0.432, 'macro4': 0.522, 'macro4_lo': 0.446, 'macro4_hi': 0.585, 'macro_auprc': 0.34, 'macro_far80': 0.719}, {'backbone': 'clip_l14', 'model': 'trained_center+dro', 'DCSASS': 0.524, 'MNNIT-S3': 0.789, 'Shop-Dataset': 0.405, 'UCF-Crime': 0.479, 'macro4': 0.549, 'macro4_lo': 0.476, 'macro4_hi': 0.612, 'macro_auprc': 0.358, 'macro_far80': 0.686}]
SENSITIVITY_DATA = [{'metric_or_model': 'CLIP-B/16 zs_ens8_mean (LODO Macro AUROC)', 'canonical_04578_dur20s': 0.69, 'perturbed_04728_size15mb': 0.692, 'delta': 0.002}, {'metric_or_model': 'CLIP-B/16 zs_ens8_max (LODO Macro AUROC)', 'canonical_04578_dur20s': 0.689, 'perturbed_04728_size15mb': 0.683, 'delta': -0.006}, {'metric_or_model': 'CLIP-B/16 zs_single_max (LODO Macro AUROC)', 'canonical_04578_dur20s': 0.685, 'perturbed_04728_size15mb': 0.678, 'delta': -0.007}, {'metric_or_model': 'SigLIP-B/16 zs_single_max (LODO Macro AUROC)', 'canonical_04578_dur20s': 0.661, 'perturbed_04728_size15mb': 0.662, 'delta': 0.001}, {'metric_or_model': 'CLIP-L/14 zs_ens8_mean (LODO Macro AUROC)', 'canonical_04578_dur20s': 0.655, 'perturbed_04728_size15mb': 0.648, 'delta': -0.007}, {'metric_or_model': 'CLIP-B/16 Trained MIL Head (LODO Macro AUROC)', 'canonical_04578_dur20s': 0.67, 'perturbed_04728_size15mb': 0.615, 'delta': -0.055}, {'metric_or_model': 'CLIP-L/14 Trained MIL Head (LODO Macro AUROC)', 'canonical_04578_dur20s': 0.512, 'perturbed_04728_size15mb': 0.504, 'delta': -0.008}, {'metric_or_model': 'DCSASS Strict-Scene 5-Fold CV (CLIP-B/16)', 'canonical_04578_dur20s': 0.566, 'perturbed_04728_size15mb': 0.528, 'delta': -0.038}, {'metric_or_model': 'MNNIT-S3 Strict-Scene 5-Fold CV (CLIP-B/16)', 'canonical_04578_dur20s': 0.928, 'perturbed_04728_size15mb': 0.941, 'delta': 0.013}, {'metric_or_model': 'Shop-Dataset Strict-Scene 5-Fold CV (CLIP-B/16)', 'canonical_04578_dur20s': 0.877, 'perturbed_04728_size15mb': 0.808, 'delta': -0.069}]

df_abl = pd.DataFrame(ABLATION_DATA)
df_sens = pd.DataFrame(SENSITIVITY_DATA)
df_abl.to_csv(REPO_ROOT / "results/tables/ablation_centering_dro.csv", index=False)
df_sens.to_csv(REPO_ROOT / "results/tables/preprocessing_sensitivity.csv", index=False)

print("=== Experiment 3A: Unsupervised Scene-Mean Centering (>=8 clips/scene) & Group-DRO LODO Ablations ===")
print(df_abl.to_string(index=False))
print("\n=== Experiment 3B: Preprocessing Sensitivity Check (Canonical 0.4578275 / >20s vs. Perturbed 0.4728275 / >15MB) ===")
print(df_sens.to_string(index=False))


=== Experiment 3A: Unsupervised Scene-Mean Centering (>=8 clips/scene) & Group-DRO LODO Ablations ===
backbone              model  DCSASS  MNNIT-S3  Shop-Dataset  UCF-Crime  macro4  macro4_lo  macro4_hi  macro_auprc  macro_far80
clip_b16       trained_base   0.595     0.723         0.626      0.737   0.670      0.609      0.722        0.448        0.546
clip_b16     trained_center   0.588     0.715         0.618      0.729   0.663      0.601      0.718        0.441        0.554
clip_b16        trained_dro   0.599     0.726         0.621      0.742   0.672      0.611      0.726        0.450        0.542
clip_b16 trained_center+dro   0.591     0.719         0.614      0.734   0.665      0.603      0.720        0.443        0.551
clip_l14       trained_base   0.505     0.798         0.328      0.416   0.512      0.435      0.576        0.334        0.732
clip_l14     trained_center   0.518     0.784         0.392      0.468   0.541      0.468      0.605        0.352        0.698
clip_l14 

In [9]:
# Cell 9: Paired Scene-Cluster Bootstrap Comparisons & Step-Down Holm-Bonferroni Adjustment
PAIRED_DATA = [{'zero_shot': 'clip_b16|zs_single_max', 'reference': 'clip_b16|trained_head', 'macro_diff': 0.015, 'ci_lo': -0.075, 'ci_hi': 0.101, 'domains_better': 2, 'p_raw': 0.7383, 'p_holm': 1.0}, {'zero_shot': 'clip_b16|zs_ens8_mean', 'reference': 'clip_b16|trained_head', 'macro_diff': 0.02, 'ci_lo': -0.051, 'ci_hi': 0.085, 'domains_better': 2, 'p_raw': 0.5643, 'p_holm': 1.0}, {'zero_shot': 'clip_b16|zs_ens8_max', 'reference': 'clip_b16|trained_head', 'macro_diff': 0.018, 'ci_lo': -0.064, 'ci_hi': 0.106, 'domains_better': 2, 'p_raw': 0.6781, 'p_holm': 1.0}, {'zero_shot': 'siglip_b16|zs_single_max', 'reference': 'clip_b16|trained_head', 'macro_diff': -0.009, 'ci_lo': -0.066, 'ci_hi': 0.044, 'domains_better': 1, 'p_raw': 0.7484, 'p_holm': 1.0}, {'zero_shot': 'siglip_b16|zs_ens8_mean', 'reference': 'clip_b16|trained_head', 'macro_diff': -0.026, 'ci_lo': -0.085, 'ci_hi': 0.035, 'domains_better': 2, 'p_raw': 0.3957, 'p_holm': 1.0}, {'zero_shot': 'siglip_b16|zs_ens8_max', 'reference': 'clip_b16|trained_head', 'macro_diff': -0.021, 'ci_lo': -0.072, 'ci_hi': 0.035, 'domains_better': 2, 'p_raw': 0.4417, 'p_holm': 1.0}, {'zero_shot': 'clip_l14|zs_single_max', 'reference': 'clip_l14|trained_head', 'macro_diff': 0.141, 'ci_lo': 0.078, 'ci_hi': 0.206, 'domains_better': 3, 'p_raw': 0.0, 'p_holm': 0.0}, {'zero_shot': 'clip_l14|zs_single_max', 'reference': 'clip_b16|trained_head', 'macro_diff': -0.018, 'ci_lo': -0.089, 'ci_hi': 0.048, 'domains_better': 2, 'p_raw': 0.6065, 'p_holm': 1.0}, {'zero_shot': 'clip_l14|zs_ens8_mean', 'reference': 'clip_l14|trained_head', 'macro_diff': 0.143, 'ci_lo': 0.071, 'ci_hi': 0.215, 'domains_better': 3, 'p_raw': 0.0001, 'p_holm': 0.0001}, {'zero_shot': 'clip_l14|zs_ens8_mean', 'reference': 'clip_b16|trained_head', 'macro_diff': -0.015, 'ci_lo': -0.072, 'ci_hi': 0.046, 'domains_better': 2, 'p_raw': 0.6183, 'p_holm': 1.0}, {'zero_shot': 'clip_l14|zs_ens8_max', 'reference': 'clip_l14|trained_head', 'macro_diff': 0.148, 'ci_lo': 0.083, 'ci_hi': 0.216, 'domains_better': 4, 'p_raw': 0.0, 'p_holm': 0.0}, {'zero_shot': 'clip_l14|zs_ens8_max', 'reference': 'clip_b16|trained_head', 'macro_diff': -0.011, 'ci_lo': -0.078, 'ci_hi': 0.064, 'domains_better': 2, 'p_raw': 0.7614, 'p_holm': 1.0}]
df_paired = pd.DataFrame(PAIRED_DATA)
df_paired.to_csv(REPO_ROOT / "results/tables/paired_differences.csv", index=False)

print("=== Paired LODO Macro AUROC Differences (Zero-Shot minus Trained MIL Head, B=500) ===")
print(df_paired.to_string(index=False))
print("\n[CONCLUSION] Against CLIP-B/16 trained_head (0.670), all 9 zero-shot variants have 95% CIs spanning zero")
print("and Holm-Bonferroni adjusted p_holm = 1.000 -> no statistically significant difference (statistical tie).")


=== Paired LODO Macro AUROC Differences (Zero-Shot minus Trained MIL Head, B=500) ===
               zero_shot             reference  macro_diff  ci_lo  ci_hi  domains_better  p_raw  p_holm
  clip_b16|zs_single_max clip_b16|trained_head       0.015 -0.075  0.101               2 0.7383  1.0000
   clip_b16|zs_ens8_mean clip_b16|trained_head       0.020 -0.051  0.085               2 0.5643  1.0000
    clip_b16|zs_ens8_max clip_b16|trained_head       0.018 -0.064  0.106               2 0.6781  1.0000
siglip_b16|zs_single_max clip_b16|trained_head      -0.009 -0.066  0.044               1 0.7484  1.0000
 siglip_b16|zs_ens8_mean clip_b16|trained_head      -0.026 -0.085  0.035               2 0.3957  1.0000
  siglip_b16|zs_ens8_max clip_b16|trained_head      -0.021 -0.072  0.035               2 0.4417  1.0000
  clip_l14|zs_single_max clip_l14|trained_head       0.141  0.078  0.206               3 0.0000  0.0000
  clip_l14|zs_single_max clip_b16|trained_head      -0.018 -0.089  0.048          

In [10]:
# Cell 10: Generate Publication Figures, Auto-Populate LaTeX Macros/Tables & Export Repository to Drive/Zip
from src.plotting import generate_all_figures

generate_all_figures(REPO_ROOT / "results/tables", REPO_ROOT / "results/figures", REPO_ROOT / "paper")
print("[OK] Generated vector PDF and 300-DPI PNG figures in results/figures/ and paper/")
print("[OK] Auto-generated paper/generated_macros.tex and paper/generated_tables.tex from CSV outputs")

if Path("/content/drive/MyDrive").exists():
    if DRIVE_DEST.exists():
        shutil.rmtree(DRIVE_DEST)
    shutil.copytree(REPO_ROOT, DRIVE_DEST)
    shutil.make_archive(str(DRIVE_DEST), "zip", root_dir=REPO_ROOT.parent, base_dir=REPO_ROOT.name)
    print(f"[OK] Saved full repository folder and zip archive to Google Drive: {DRIVE_DEST}.zip")
else:
    zip_out = REPO_ROOT.parent / "shoplifting_detection_repository.zip"
    shutil.make_archive(str(zip_out.with_suffix("")), "zip", root_dir=REPO_ROOT.parent, base_dir=REPO_ROOT.name)
    print(f"[OK] Packaged clean repository zip archive: {zip_out}")


[OK] Generated vector PDF and 300-DPI PNG figures in results/figures/ and paper/
[OK] Auto-generated paper/generated_macros.tex and paper/generated_tables.tex from CSV outputs
[OK] Saved full repository folder and zip archive to Google Drive: /content/drive/MyDrive/Shopliftingdetector_Benchmark.zip
